In [4]:
import pandas as pd

In [ ]:

import os
print(os.getcwd())

/home/ayxiao227/dft_ml_project/notebooks


In [19]:
constraints = {
    "max_molecular_weight": 150,
    "max_heavy_atoms": 10,
    "allowed_elements": {"C", "H", "O", "N"},
    "neutral_only": True
}

In [13]:
df = pd.read_csv("/home/ayxiao227/dft_ml_project/molecules/csv_files/pubchem_molecules_1.csv")

In [14]:
df

,Compound_CID,IUPAC_Name,SMILES,Heavy_Atom_Count,Molecular_Weight,Molecular_Formula,Charge,Name
0,4,1-aminopropan-2-ol,CC(CN)O,5,75.11,C3H9NO,0,1-Amino-2-propanol
1,11,"1,2-dichloroethane",C(CCl)Cl,4,98.96,C2H4Cl2,0,"1,2-Dichloroethane"
2,33,2-chloroacetaldehyde,C(C=O)Cl,4,78.50,C2H3ClO,0,Chloroacetaldehyde
3,34,2-chloroethanol,C(CCl)O,4,80.51,C2H5ClO,0,2-Chloroethanol
4,75,3-aminopropanal,C(CN)C=O,5,73.09,C3H7NO,0,"Propanal, 3-amino-"
...,...,...,...,...,...,...,...,...
107956,179062796,N-methoxysulfanylmethanimine,COSN=C,5,91.13,C2H5NOS,0,CID 179062796
107957,179070167,NaN,[B](C)C/C=C\C,6,80.95,C5H10B,0,CID 179070167
107958,179070950,NaN,[B]C(=C)/N=C(\C)/F,7,96.90,C4H5BFN,0,CID 179070950
107959,179072105,"1-[(1R,2S)-2-methylcyclopropyl]ethanamine",C[C@H]1C[C@H]1C(C)N,7,99.17,C6H13N,0,CID 179072105


In [16]:
valid_df = df[
    (df["Charge"] == 0)
    & (df["Heavy_Atom_Count"] < 10)
    & (df["Molecular_Weight"] < 150)
]

In [17]:
valid_df

,Compound_CID,IUPAC_Name,SMILES,Heavy_Atom_Count,Molecular_Weight,Molecular_Formula,Charge,Name
0,4,1-aminopropan-2-ol,CC(CN)O,5,75.11,C3H9NO,0,1-Amino-2-propanol
1,11,"1,2-dichloroethane",C(CCl)Cl,4,98.96,C2H4Cl2,0,"1,2-Dichloroethane"
2,33,2-chloroacetaldehyde,C(C=O)Cl,4,78.50,C2H3ClO,0,Chloroacetaldehyde
3,34,2-chloroethanol,C(CCl)O,4,80.51,C2H5ClO,0,2-Chloroethanol
4,75,3-aminopropanal,C(CN)C=O,5,73.09,C3H7NO,0,"Propanal, 3-amino-"
...,...,...,...,...,...,...,...,...
107956,179062796,N-methoxysulfanylmethanimine,COSN=C,5,91.13,C2H5NOS,0,CID 179062796
107957,179070167,NaN,[B](C)C/C=C\C,6,80.95,C5H10B,0,CID 179070167
107958,179070950,NaN,[B]C(=C)/N=C(\C)/F,7,96.90,C4H5BFN,0,CID 179070950
107959,179072105,"1-[(1R,2S)-2-methylcyclopropyl]ethanamine",C[C@H]1C[C@H]1C(C)N,7,99.17,C6H13N,0,CID 179072105


In [18]:
import regex as re
def get_elements_from_smiles_regex(smiles):
    # bracket atoms: [ ... ]  |  two-letter organic subset: Br, Cl  |  single-letter (incl. aromatic)
    token_re = re.compile(r'\[([^\]]+)\]|(Br|Cl|[BCNOPSFI]|[bcnops])')

    elements = set()
    for m in token_re.finditer(smiles):
        bracket_content, plain = m.group(1), m.group(2)
        if bracket_content is not None:
            # inside brackets: optional isotope digits, then the element symbol
            mm = re.match(r'^\d*([A-Za-z][a-z]?)', bracket_content)
            if mm:
                sym = mm.group(1)
                elements.add(sym.capitalize())
        else:
            elements.add(plain.capitalize())
    return elements

In [21]:
valid_df = valid_df[
    valid_df["SMILES"].apply(
        lambda smiles:
            get_elements_from_smiles_regex(smiles)
            .issubset(constraints["allowed_elements"])
    )
]

In [22]:
valid_df

,Compound_CID,IUPAC_Name,SMILES,Heavy_Atom_Count,Molecular_Weight,Molecular_Formula,Charge,Name
0,4,1-aminopropan-2-ol,CC(CN)O,5,75.11,C3H9NO,0,1-Amino-2-propanol
4,75,3-aminopropanal,C(CN)C=O,5,73.09,C3H7NO,0,"Propanal, 3-amino-"
5,118,4-aminobutanal,C(CC=O)CN,6,87.12,C4H9NO,0,4-Aminobutanal
6,174,"ethane-1,2-diol",C(CO)O,4,62.07,C2H6O2,0,Ethylene Glycol
8,176,acetic acid,CC(=O)O,4,60.05,C2H4O2,0,Acetic Acid
...,...,...,...,...,...,...,...,...
107951,179042436,"4-ethenyl-3-oxido-1,4-dihydrotriazet-3-ium",C=CC1NN=[N+]1[O-],7,99.09,C3H5N3O,0,CID 179042436
107952,179043761,2-methyloxirane;molecular nitrogen,CC1CO1.N#N,6,86.09,C3H6N2O,0,CID 179043761
107954,179054642,(1R)-1-methyl-3-oxabicyclo[3.1.0]hexane,C[C@@]12CC1COC2,7,98.14,C6H10O,0,CID 179054642
107959,179072105,"1-[(1R,2S)-2-methylcyclopropyl]ethanamine",C[C@H]1C[C@H]1C(C)N,7,99.17,C6H13N,0,CID 179072105
